# Time one standard 158M training run

Select the RTX PRO 6000 GPU in Colab, run the cells in order, and approve the Google Drive mount when prompted.


In [ ]:
# Experiment storage
USE_GOOGLE_DRIVE = True

from pathlib import Path
import os

default_workspace = "/content/bmr" if Path("/content").is_dir() else "~/bmr-workspace"
WORKSPACE = Path(os.environ.get("BMR_WORKSPACE", default_workspace)).expanduser().resolve()
EXPERIMENT = "standard_vs_gated_160m_tinyllama"
DATA = WORKSPACE / "data" / "fineweb_edu_tinyllama"
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    my_drive = Path("/content/drive/MyDrive")
    if not my_drive.is_dir():
        raise RuntimeError("Mount Google Drive before continuing.")
    storage = my_drive / "bmr"
else:
    storage = WORKSPACE
RUNS = storage / "runs" / EXPERIMENT
SAVED_DATA = storage / "data" / "fineweb_edu_tinyllama"
print("Experiment outputs:", RUNS)


In [ ]:
# Set up experiment runtime
"""Standard-library setup embedded in the notebooks so it can run before the checkout exists."""

import fcntl
import json
import re
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

GITHUB_REPO_URL = "https://github.com/seungjun-green/base-model-research"


def _checkout(workspace, runs):
    source_file = runs / "source.json"
    if source_file.exists():
        source = json.loads(source_file.read_text())
        if source["url"].rstrip("/").removesuffix(".git").lower() != GITHUB_REPO_URL.lower():
            raise ValueError(
                f"{runs} is already pinned to a different repository. Use a new EXPERIMENT name."
            )
        commit = source["commit"]
    else:
        head = subprocess.check_output(
            ["git", "ls-remote", "--exit-code", GITHUB_REPO_URL, "HEAD"], text=True
        )
        commit = head.split()[0]
        source = {"url": GITHUB_REPO_URL, "commit": commit}
    if not re.fullmatch(r"[0-9a-f]{40}", commit):
        raise ValueError(f"Invalid Git commit for this experiment: {commit!r}")

    repo = workspace / "repos" / commit
    if not repo.exists():
        repo.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--", GITHUB_REPO_URL, repo], check=True)
    if subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip():
        raise RuntimeError(f"The runtime checkout has local edits: {repo}")
    if subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip() != commit:
        subprocess.run(["git", "fetch", "origin", commit], cwd=repo, check=True)
        subprocess.run(["git", "checkout", "--detach", commit], cwd=repo, check=True)
    if not source_file.exists():
        temporary = source_file.with_suffix(".tmp")
        temporary.write_text(json.dumps(source, indent=2) + "\n")
        temporary.replace(source_file)
    return repo


def setup_colab(workspace, runs, *, benchmark):
    runs.mkdir(parents=True, exist_ok=True)
    with (runs / ".setup.lock").open("a") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX)
        repo = _checkout(workspace, runs)
        try:
            torch_version = version("torch")
        except PackageNotFoundError as error:
            raise RuntimeError("Select a Colab runtime with PyTorch installed and rerun setup.") from error

        # Keep the runtime's PyTorch build, including its CUDA version suffix.
        extras = "data,eval" if benchmark else "data"
        print(f"Using {sys.executable} with PyTorch {torch_version}", flush=True)
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                f"torch=={torch_version}",
                "-e",
                f"{repo}[{extras}]",
            ],
            check=True,
        )
    return repo

REPO = setup_colab(WORKSPACE, RUNS, benchmark=False)

def run(*arguments):
    subprocess.run([sys.executable, "-m", "base_model_research.cli", *map(str, arguments)],
                   cwd=REPO, check=True)


Run this once the TinyLlama-tokenized FineWeb-Edu corpus from notebook 01 is ready. It draws one of the 32 hyperparameter combinations and trains on 3,156,295,680 targets (20 per parameter). You can run it before the proxy searches. The tqdm bar estimates the remaining time from completed optimizer updates. Give it several updates to settle. Validation and Drive writes can add time beyond the early estimate.


In [ ]:
import random
import sys

sys.path.insert(0, str(REPO / "src"))
from base_model_research.config import Hyperparameters, ModelConfig, TrainConfig
from base_model_research.search import load_trials

DRAW_SEED = 42  # The same seed selects the same hyperparameters when you reopen Colab.
trials = load_trials(REPO / "experiments/trials_seed42.json")["trials"]
trial = random.Random(DRAW_SEED).choice(trials)
model = ModelConfig.load(REPO / "configs/models/standard_main.yaml")
hp = Hyperparameters(**trial["hyperparameters"])
output = RUNS / "timing" / "standard" / trial["id"]
record = output / "run.json"
profile = RUNS / "standard_main_profile.json"
if record.is_file():
    settings = TrainConfig.from_record(json.loads(record.read_text())["training"])
else:
    memory = json.loads(profile.read_text())["training"] if profile.is_file() else {}
    settings = TrainConfig.load(REPO / "configs/training.yaml",
                                microbatch_size=memory.get("microbatch_size"),
                                activation_checkpointing=memory.get("activation_checkpointing"))

print(f"Selected {trial['id']} with a microbatch of {settings.microbatch_size} sequences.")
print(json.dumps(trial["hyperparameters"], indent=2))
print(f"Timing run folder: {output}")


You can stop the next cell once you have a time estimate, or leave it running to finish. Each of the 20 validations saves weights and optimizer state to the timing run folder (about 38 GB total for a full run). Rerunning resumes from the latest completed save. Before the first save, it restarts from the original seed.


In [ ]:
from contextlib import chdir, ExitStack
from base_model_research.training import train
from tqdm.auto import tqdm

def train_one_model():
    if not (SAVED_DATA / "manifest.json").is_file():
        raise FileNotFoundError(f"Prepare and upload the notebook 01 corpus to {SAVED_DATA}.")
    cache = DATA.parent / "cached_shards" if DATA != SAVED_DATA else None
    with chdir(REPO), ExitStack() as display:
        bar = None

        def show_update(label, unit, completed, total, detail=""):
            nonlocal bar
            if bar is None:
                bar = display.enter_context(tqdm(desc=label, unit=unit, initial=completed, total=total))
            bar.set_postfix_str(detail, refresh=False)
            bar.update(completed - bar.n)

        return train(model, hp, settings, SAVED_DATA, output, resume=record.is_file(),
                     selection={"method": "random", "trial_id": trial["id"]},
                     progress=show_update, cache_dir=cache)

print("Initializing the standard model. The ETA appears after the first optimizer update.", flush=True)
result = train_one_model()
print(f"Run {result['status']} at update {result['step']:,}.")
